In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder 
  .appName("MinIO Write") 
  .master("local[*]") 
  .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
  .config("spark.log.level", "ERROR") 
  .config("spark.eventLog.enabled", "false") 
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") 
  .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog") 
  .config("spark.sql.catalog.icebergtb.type", "hadoop") 
  .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/") 
  .config("spark.sql.defaultCatalog", "icebergtb") 
  .config("spark.jars.packages", 
          "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.261,"
          "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
          "com.github.ben-manes.caffeine:caffeine:3.1.8,org.apache.parquet:parquet-hadoop:1.13.1") 
  .config("spark.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000") 
  .config("spark.hadoop.fs.s3a.access.key", "minioadmin") 
  .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") 
  .config("spark.hadoop.fs.s3a.path.style.access", "true") 
  .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
  .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
  .getOrCreate()
)

# create sample dataframe
data = [("Alice", 29), ("Bob", 35), ("Catherine", 24), ("novoteste", 30)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)

# show dataframe data
df.show()

# write on minio
output_path = "s3a://testespark/notebook/"  
df.repartition(10).write.mode("overwrite").parquet(output_path)

print(f"Data written successfully to {output_path}")

df = spark.read.parquet(f"{output_path}")

df.show()

In [ ]:
data = [("Alice", 29), ("Bob", 35), ("Catherine", 24), ("novoteste", 30)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)

# show dataframe data
df.show()

# write on minio
output_path = "s3a://testespark/novo_notebook/"  
df.write.mode("overwrite").parquet(output_path)

print(f"Data written successfully to {output_path}")

df = spark.read.parquet(f"{output_path}")

df.show()

In [ ]:
# Liste as tabelas no catálogo e esquema
spark.sql("SHOW TABLES IN icebergtb.advworks_rawdb").show()

In [ ]:
spark.sql("select*from s3a://iceberg/advworks_rawdb/raw_dimbumyer""")

In [ ]:
# Consulta simples
result = spark.sql("SELECT * FROM icebergtb.advworks_rawdb.raw_dimbuyer")
display(result)  # Ou result.show()

# Listar tabelas no esquema
spark.sql("SHOW TABLES IN icebergtb.advworks_rawdb").show()


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Iceberg with MinIO")
    .master("local[*]")
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
            "org.scala-lang:scala-library:2.13.12")
        # Configuração do Catálogo Iceberg
        .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
        .config("spark.log.level", "ERROR") 
        .config("spark.eventLog.enabled", "false") 
        # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
        .config("spark.sql.catalog.icebergtb.type", "hadoop") 
        .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
        .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://localhost:9000")
        .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
        .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
    .getOrCreate()
)

spark

In [ ]:
# Liste as tabelas no catálogo e esquema
spark.sql("SHOW TABLES IN icebergtb.advworks_rawdb").show()

In [ ]:
# Consulta simples
result = spark.sql("SELECT * FROM icebergtb.advworks_rawdb.raw_dimbuyer")
display(result)  # Ou result.show()

In [ ]:
result = spark.sql(""" SELECT 
                            
                            businessentityid,
                            firstname,
                            lastname
                       FROM icebergtb.advworks_rawdb.raw_dimbuyer""")

result.show(2, truncate=False)


# spark.sql(""" select 
#                     committed_at,
#                     snapshot_id,
#                     parent_id
                    
#                 from icebergtb.advworks_rawdb.raw_dimbuyer.stats""").show(truncate=False)

In [ ]:
# spark.sql(""" select * from icebergtb.advworks_rawdb.raw_dimcustomer""").show()


spark.sql(""" 
                select 
                    rdc.customerid,
                    rdc.firstname,
                    rdc.lastname
                    
                from icebergtb.advworks_rawdb.raw_dimcustomer as rdc
                    inner join
                    
                    icebergtb.advworks_rawdb.raw_dimbuyer as rdb
                on rdc.businessentityid = rdb.businessentityid
            
        """).count()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col
df = spark.sql("select * FROM icebergtb.advworks_rawdb.raw_dimbuyer")

df2 = spark.sql("select * from icebergtb.advworks_rawdb.raw_dimcustomer")



# Realizando o join utilizando a coluna businessentityid com um alias para evitar duplicação
df_final = df.alias("df1").join(
    df2.alias("df2"),
    col("df1.businessentityid") == col("df2.businessentityid"),
    "left"
)
df_final.show()

In [ ]:
from pyspark.sql.session import SparkSession

spark = (
            SparkSession
                .builder
                .master("spark://DOUG.:7077")

                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://localhost:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark